# Teste de Inferência - Llama 3.1 8B no Google Colab (T4)

Testa velocidade de inferência de postura parlamentar usando GPU.
Compara com os 11s/inferência do Mac M1 Air local.

## 1. Upload dos arquivos

In [ ]:
from google.colab import files
import zipfile

print("📁 Faça upload do ZIP com a estrutura:")
print("   benchmark-llm/")
print("   ├── proposicoes/textos/pl_001.txt até pl_004.txt")
print("   ├── discursos/pl_001_a_favor.txt até pl_004_contra.txt")
print("   └── resultados/resumos_llama31_8b.json\n")

uploaded = files.upload()

for filename in uploaded.keys():
    if filename.endswith('.zip'):
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('.')
        print(f"✓ {filename} descompactado")

## 2. Instala dependências

In [ ]:
!pip install -q transformers torch accelerate bitsandbytes

## 3. Carrega o modelo

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import json, time
from pathlib import Path

print("🔧 Configurando modelo Llama 3.1 8B para GPU T4...")

# CORREÇÃO 1: A T4 exige float16. bfloat16 nela causa lentidão extrema.
compute_dtype = torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype
)

model_id = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# CORREÇÃO 2: Configurar padding correto para permitir envio em lotes (batches)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=compute_dtype,
    attn_implementation="sdpa" # CORREÇÃO 3: Ativa a atenção otimizada do PyTorch 2.0
)
print("✓ Modelo otimizado carregado!\n")

## 4. Carrega os dados

In [ ]:
# Verifica estrutura de diretórios
import os

print("📁 Verificando estrutura:")
for root, dirs, files in os.walk("."):
    level = root.replace(".", "", 1).count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = " " * 2 * (level + 1)
    for file in files[:5]:  # mostra só primeiros 5 arquivos
        print(f"{subindent}{file}")
    if len(files) > 5:
        print(f"{subindent}... e mais {len(files)-5} arquivos")

In [ ]:
# Carrega dados (estão na raiz, não em benchmark-llm/)
from pathlib import Path

BASE_DIR = Path(".")  # mudou aqui
ementas, discursos = {}, {}

for txt_file in (BASE_DIR / "proposicoes/textos").glob("pl_*.txt"):
    with open(txt_file, encoding="utf-8") as f:
        for linha in f.read().split("\n"):
            if linha.strip() and not linha.startswith("EMENTA:"):
                ementas[txt_file.stem] = linha.strip()
                break

for disc_file in (BASE_DIR / "discursos").glob("pl_*_*.txt"):
    with open(disc_file, encoding="utf-8") as f:
        discursos[disc_file.stem] = f.read()

print(f"✓ {len(ementas)} ementas, {len(discursos)} discursos\n")

## 5. Função de inferência

In [ ]:
import time
import json
import torch
import re

print("="*60)
print("  TESTE - GPU T4 (PROCESSAMENTO EM LOTE + CoT OTIMIZADO + REGEX)")
print("="*60)

BATCH_SIZE = 8 # Aumente ou diminua dependendo do uso da VRAM
resultados = []

# Prepara a lista completa de tarefas
tarefas = []
for caso_id, discurso in sorted(discursos.items()):
    partes = caso_id.split("_")
    pl_id, gabarito = f"{partes[0]}_{partes[1]}", " ".join(partes[2:]).upper()
    ementa = ementas.get(pl_id)
    if ementa:
        tarefas.append({"caso_id": caso_id, "ementa": ementa, "discurso": discurso, "gabarito": gabarito})

start_total = time.time()

# Processa em blocos (batches)
for i in range(0, len(tarefas), BATCH_SIZE):
    lote = tarefas[i : i + BATCH_SIZE]
    prompts = []

    for t in lote:
        prompt = f"""Você é um analista político especializado em inferir posicionamentos de parlamentares.

Sua tarefa: analisar um discurso histórico de um deputado e inferir se ele votaria A FAVOR ou CONTRA da proposição descrita abaixo.

IMPORTANTE:
- O discurso NÃO menciona a proposição explicitamente
- Você deve INFERIR a postura com base nos valores, argumentos e posicionamentos do deputado
- Responda APENAS em formato JSON válido, sem texto adicional

PROPOSIÇÃO (resumo):
{t['ementa']}

---

DISCURSO DO DEPUTADO (histórico):
{t['discurso']}

---

Baseado no discurso acima, o deputado votaria A FAVOR ou CONTRA desta proposição?

Responda em JSON seguindo EXATAMENTE esta ordem (pense na justificativa antes de definir a postura final):
{{
    "justificativa": "Explique em no MÁXIMO 15 palavras a razão da postura",
    "postura": "A FAVOR" ou "CONTRA"
}}

JSON:"""
        prompts.append(prompt)

    # Tokeniza todos os prompts do lote
    inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=1024).to(model.device)

    # Configura os tokens de parada oficiais do Llama 3 Instruct
    terminators = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|eot_id|>")
    ]

    start_lote = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,  # Teto reduzido para velocidade máxima
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=terminators # Força a parada imediata
        )
    tempo_lote = time.time() - start_lote
    tempo_por_item = tempo_lote / len(lote)

    # Decodifica e salva os resultados
    for j, output in enumerate(outputs):
        resposta = tokenizer.decode(output[inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        t = lote[j]

        try:
            # EXTRAÇÃO BLINDADA: Pega estritamente o que está entre a primeira { e a última }
            match = re.search(r'\{.*?\}', resposta, re.DOTALL)

            if match:
                clean = match.group(0)
                resultado_json = json.loads(clean)
                postura = resultado_json.get("postura", "").upper()
                justif = resultado_json.get("justificativa", "")
            else:
                raise ValueError("JSON não encontrado na resposta.")

        except Exception as e:
            # Fallback cirúrgico caso tudo dê errado
            postura = "CONTRA" if '"postura": "CONTRA"' in resposta else ("A FAVOR" if '"postura": "A FAVOR"' in resposta else "INDEFINIDO")
            justif = resposta[:200]

        acertou = (postura == t['gabarito'])
        print(f"{t['caso_id']:<20} → {'✓' if acertou else '✗'} {postura} ({tempo_por_item:.2f}s/item)")

        resultados.append({
            "caso": t['caso_id'], "gabarito": t['gabarito'],
            "inferido": postura, "acertou": acertou,
            "tempo_s": round(tempo_por_item, 2), "justificativa": justif
        })

print("\n" + "="*60)
tempo_total = time.time() - start_total
acertos = sum(r["acertou"] for r in resultados)
tempo_medio = tempo_total / len(resultados) if resultados else 0

print(f"Acurácia:    {acertos}/{len(resultados)} ({acertos/len(resultados)*100:.1f}%)")
print(f"Tempo médio real: {tempo_medio:.2f}s por inferência")
print(f"Speedup vs Mac:   {19/tempo_medio:.1f}x")
print(f"Projeção: {42500} inferências = {42500*tempo_medio/3600:.1f}h na T4")